# Data Preprocessing and Missing-Value Treatment
### A comparative study using residential property data

**Academic group project** | Coimbra Business School (ISCAC), Polytechnic Institute of Coimbra  
**Contributors:** Luís Coimbra, Mário Simão and Ricardo Neto  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Missingno · Scikit-learn

---

## Project overview

This notebook examines how different strategies for handling missing data affect the information retained in a residential property dataset. It combines exploratory data analysis, missing-data visualization, deletion strategies and three imputation approaches.

**Case study:** Historical residential property sales in King County, Washington, from May 2014 to May 2015. The scenario considers an online real estate company that relies on reliable property data for valuation models.

**Methods compared**
- Column removal, complete-case deletion and available-case filtering
- Median imputation
- Linear regression imputation
- K-nearest neighbors (KNN) imputation, with five neighbors

> **Academic context:** This is an English-language portfolio edition of the original group coursework. The underlying analytical workflow and most of the Python code have been preserved. Minor corrections improve compatibility and fix an incorrectly fitted regression model.

> **Dataset:** The dataset (dataset.csv) was provided during our university classes for academic purposes and has not been publicly released. Therefore, it is not included in this repository.


## 1. Dataset and environment

Import the required libraries and load the property-sales dataset.

In [ ]:
# Core data analysis and visualization libraries
import numpy as np
import pandas as pd
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.impute import KNNImputer

pd.set_option("display.float_format", lambda value: f"{value:.2f}")

In [ ]:
# Load the source dataset (place dataset.csv next to this notebook)
df = pd.read_csv("dataset.csv")

### Business context

The dataset describes historical residential property transactions in King County, Washington, USA. The practical objective is to prepare the data for reliable downstream analysis and potential property valuation models. Missing or inconsistent values can reduce the quality of statistical conclusions and model predictions.

The original coursework describes **1,639 observations and 12 variables**. The next cell checks the actual dimensions of the supplied CSV.

In [ ]:
rows, columns = df.shape
print("Number of rows:", rows)
print("Number of columns:", columns)

### Initial inspection

Display the first ten records to inspect the structure and identify potential missing values. The original analysis highlighted missing entries in `grade` and `sqft_basement`; subsequent checks also identified missing values in `zipcode`.

In [ ]:
df.head(10)

## 2. Exploratory data analysis

### Column-level profiling

Use descriptive statistics to inspect the numeric variables, including their non-null counts, central tendencies and ranges.

- **`id`:** Count unique identifiers rather than interpreting the identifier as a numeric measurement.
- **`zipcode`:** Inspect distinct categories and the number of populated entries.
- **Other variables:** Use `describe()` to understand distributions and detect differences in non-null counts.

The original analysis focuses on missing entries in `grade`, `sqft_basement` and `zipcode`.

In [ ]:
unique_ids = df["id"].nunique()
print("Number of unique IDs:", unique_ids)

In [ ]:
df['price'].describe()

In [ ]:
df['bedrooms'].describe()

In [ ]:
df['bathrooms'].describe()

In [ ]:
df['sqft_living'].describe()

In [ ]:
df['sqft_lot'].describe()

In [ ]:
df['grade'].describe()

In [ ]:
df['sqft_above'].describe()

In [ ]:
df['sqft_basement'].describe()

In [ ]:
df['yr_built'].describe()


In [ ]:
df['yr_renovated'].describe()

In [ ]:
unique_zipcodes = df["zipcode"].nunique()
print("Number of unique ZIP codes:", unique_zipcodes)

zipcode_count = df["zipcode"].count()
print("Number of non-missing ZIP codes:", zipcode_count)

## 3. Missing-data assessment

### Completeness by variable

The `missingno` bar chart shows the number of populated values in each column, making incomplete variables easier to identify.

In [ ]:
msno.bar(df, color='g')

### Missingness relationships

A missingness heatmap visualizes relationships between patterns of missing entries. The original analysis did not identify a strong visible relationship. **A heatmap alone cannot establish whether data are MCAR, MAR or MNAR**; additional assumptions or statistical investigation would be required.

In [ ]:
msno.heatmap(df)

## 4. Missing-data deletion strategies

Deletion can be simple, but it may discard useful information. The following sections preserve the original three approaches for comparison.

### 4.1 Remove incomplete columns

Drop any variable containing at least one missing value. This produces a complete set of retained columns, but potentially sacrifices important predictors.

In [ ]:
columns_removed = df.dropna(axis="columns")
print("Dataset after removing columns containing missing values:")
columns_removed

### 4.2 Complete-case analysis (listwise deletion)

Keep only records without missing values in any column. This approach may substantially reduce the available sample.

In [ ]:
complete_cases = df.dropna(how="any")
print("Dataset containing only complete rows:")
complete_cases

### 4.3 Available-case filtering

Retain rows where at least one of the three selected variables (`grade`, `sqft_basement`, `zipcode`) is present. **Note:** This is the original notebook's filtering rule, not a general implementation of pairwise deletion for statistical calculations.

In [ ]:
available_cases = df.dropna(
    subset=["grade", "sqft_basement", "zipcode"], how="all"
)
available_cases

## 5. Missing-value imputation

Imputation retains rows by estimating or substituting missing entries. The methods below are compared as alternative treatments of the same source data.

### Handling the categorical field

`zipcode` is treated as a categorical identifier. The original project fills its missing entries with the **most frequent observed value (mode)** for all three imputation approaches. The numeric variables `grade` and `sqft_basement` are handled using the methods described below.

> Mode imputation is a simple baseline; it does not guarantee recovery of the original missing values.

### 5.1 Median imputation

Replace missing numeric values with each column's observed median. This provides a straightforward baseline that is less sensitive to outliers than mean imputation.

In [ ]:
df_central = df.copy()
df_central["grade"] = df_central["grade"].fillna(df_central["grade"].median())
df_central["sqft_basement"] = df_central["sqft_basement"].fillna(
    df_central["sqft_basement"].median()
)
df_central["zipcode"] = df_central["zipcode"].fillna(
    df_central["zipcode"].mode()[0]
)
df_central

In [ ]:
unique_zipcodes = df_central["zipcode"].nunique()
central_zipcode_count = df_central["zipcode"].count()
central_grade_count = df_central["grade"].count()
central_basement_count = df_central["sqft_basement"].count()

print("Unique ZIP codes after imputation:", unique_zipcodes)
print("Non-missing ZIP codes:", central_zipcode_count)
print("Non-missing grade values:", central_grade_count)
print("Non-missing basement values:", central_basement_count)

The following counts verify that the selected columns have been populated after median/mode imputation.

### 5.2 Linear regression imputation

Fit regression models on available observations and use their predictions to fill missing numeric values. Existing non-missing values are retained.

### Predictor selection

The original analysis selected `sqft_above` to predict missing `grade` values and `sqft_living` to predict missing `sqft_basement` values. A correlation heatmap provides exploratory context for these choices.

**Small correction:** The basement regression must be fitted using `sqft_living` and `sqft_basement`, rather than accidentally reusing the grade model's training data.

In [ ]:
# Correlations between numeric variables
corr_matrix = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix - Numeric Property Features")
plt.tight_layout()
plt.show()

In [ ]:
df_linear = df.copy()

In [ ]:
# Fit each model using rows where its target and predictor are available
grade_training = df_linear.dropna(subset=["grade", "sqft_above"])
basement_training = df_linear.dropna(subset=["sqft_basement", "sqft_living"])

In [ ]:
X = grade_training[["sqft_above"]]
y = grade_training["grade"]
model = LinearRegression().fit(X, y)

In [ ]:
a = basement_training[["sqft_living"]]
b = basement_training["sqft_basement"]
model1 = LinearRegression().fit(a, b)

In [ ]:
# Predict only the missing values, keeping observed values unchanged
missing_grade = df_linear["grade"].isna() & df_linear["sqft_above"].notna()
df_linear.loc[missing_grade, "grade"] = model.predict(
    df_linear.loc[missing_grade, ["sqft_above"]]
)

missing_basement = (
    df_linear["sqft_basement"].isna() & df_linear["sqft_living"].notna()
)
df_linear.loc[missing_basement, "sqft_basement"] = model1.predict(
    df_linear.loc[missing_basement, ["sqft_living"]]
)

In [ ]:
df_linear["zipcode"] = df_linear["zipcode"].fillna(
    df_linear["zipcode"].mode()[0]
)
df_linear

### 5.3 K-nearest neighbors (KNN) imputation

Apply `KNNImputer` to `grade` and `sqft_basement` using **five nearest neighbors**, then use mode imputation for `zipcode`.

The original two-column feature selection is preserved. Because KNN is distance-based, feature scaling and the choice of predictors would deserve further evaluation in a production setting.

In [ ]:
df_knn = df.copy()
imputer = KNNImputer(n_neighbors=5)

knn_cols = ["grade", "sqft_basement"]
df_knn[knn_cols] = imputer.fit_transform(df_knn[knn_cols])

mode_col = "zipcode"
df_knn[mode_col] = df_knn[mode_col].fillna(df_knn[mode_col].mode()[0])
df_knn

In [ ]:
df_knn['grade'].describe()

## 6. Comparing the imputation results

The final section compares the datasets created by the three imputation methods. It checks whether the target fields are populated and examines descriptive statistics for `grade` and `sqft_basement`.

These comparisons are **descriptive**: they do not establish which imputation method is most accurate without a validation procedure using known or artificially masked values.

In [ ]:
df_central['grade'].describe()

In [ ]:
df_central['sqft_basement'].describe()

In [ ]:
df_central['zipcode'].count()

In [ ]:
df_knn['grade'].describe()

In [ ]:
df_knn['sqft_basement'].describe()

In [ ]:
df_knn['zipcode'].count()

In [ ]:
df_linear['grade'].describe()

In [ ]:
df_linear['sqft_basement'].describe()

In [ ]:
df_linear['zipcode'].count()